email agent
- authenticates user
    - only then are they allowed into the "inbox"
    - dynamic tools and prompt on the condition of there being an email and password in state that match hardcoded
- checks "inbox"
    - email in tool
- sends emails
    - human in the loop

In [1]:
import warnings
warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*", category=UserWarning)

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "julie@example.com"
    password: str = "password123"

In [3]:
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

In [4]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for recent emails"""
    return """
    Hi Julie, 
    I'm going to be in town next week and was wondering if we could grab a coffee?
    - best, Jane (jane@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send an response email"""
    return f"Email sent to {to} with subject {subject} and body {body}"

@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the user with the given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(update={
            "authenticated": True, 
            "messages": [ToolMessage(
                "Successfully authenticated", 
                tool_call_id=runtime.tool_call_id)]
        })
    else:
        return Command(update={
            "authenticated": False,
            "messages": [ToolMessage(
                "Authentication failed", 
                tool_call_id=runtime.tool_call_id)]
        })

In [5]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Allow read inbox and send email tools only if user provides correct email and password"""

    authenticated = request.state.get("authenticated")
    
    if authenticated:
        tools = [check_inbox, send_email]
    else:
        tools = [authenticate]

    request = request.override(tools=tools) 
    return handler(request)

>Note: the prompts were modified since filming to constrain the model to more reliably match the filmed sequence. You may still experience different responses from the model, which is expected. You may need to modify the human message to provide appropriate responses.

In [6]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = """You are a helpful assistant that can check the inbox and send emails. 
Your first step after authentication is to check the inbox."""
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."

@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [7]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    "groq:openai/gpt-oss-20b",
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call, 
        dynamic_prompt,
        HumanInTheLoopMiddleware[AgentState, EmailContext](
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            })
        ]
    )


In [8]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="julie@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

Got it! I’ve fetched your inbox and found one recent email from Jane:

> **Hi Julie,**  
> I'm going to be in town next week and was wondering if we could grab a coffee?  
> - best, Jane (jane@example.com)

How would you like to proceed?  
- Reply to Jane  
- Delete the email  
- Mark as read  
- Anything else?


In [10]:

response = agent.invoke(
    {"messages": [HumanMessage(content="any draft is fine. don't check back.")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

Here’s a simple draft you can use (just copy‑paste it into your email client):

---

**Subject:** Re: Coffee next week

Hi Jane,

Thanks for reaching out! I’d love to catch up. How about [suggest a date/time] next week?

Looking forward to it!

- Julie

---


In [17]:
# Trigger a send_email approval step, then inspect the interrupt payload only if it exists
response = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content="Send a short follow-up email to jane@example.com asking to meet for coffee next week."
            )
        ]
    },
    context=EmailContext(),
    config=config,
)

if "__interrupt__" in response:
    pending = response["__interrupt__"][0].value
    body = pending["action_requests"][0]["args"]["body"]
    print(body)
else:
    print("No interrupt was raised; the agent did not pause for approval.")
    print(response["messages"][-1].content)

Hi Jane,

Just checking in—would love to meet for coffee next week. Let me know what works for you.

Thanks!
Julie


In [20]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ), 
    config=config # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

Got it! I’ve sent the follow‑up email to Jane. Let me know if there’s anything else you’d like to do.


In [ ]:
from pprint import pprint

pprint(response)      

{'authenticated': True,
 'messages': [HumanMessage(content='julie@example.com, password123', additional_kwargs={}, response_metadata={}, id='251b4a0d-b201-4f22-999c-23fec87d94a4'),
              AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants to authenticate. We need to call function authenticate.', 'tool_calls': [{'id': 'fc_66b69664-229c-4ab6-8f47-a83e901cb945', 'function': {'arguments': '{"email":"julie@example.com","password":"password123"}', 'name': 'authenticate'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 146, 'total_tokens': 189, 'completion_time': 0.069920494, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.008014289, 'prompt_tokens_details': None, 'queue_time': 0.01627793, 'total_time': 0.077934783}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_8d13edce1d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'g